In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install cyvcf2

!apt-get update
!apt-get install -y bcftools tabix

input_gnomad = "/content/drive/MyDrive/FYP_DATA/raw_data/gnomad_data/joint_vcf/chr5.vcf.bgz"

reference_fasta = "/content/drive/MyDrive/FYP_DATA/raw_data/reference_fasta/hg38_dna.fa"

output_dir = "/content/drive/MyDrive/FYP_DATA/processed_data/gnomad"


from cyvcf2 import VCF
import subprocess
import os
import pandas as pd
import numpy as np
from cyvcf2 import VCF

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.6 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,201 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,519 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/p

In [4]:
# for chr5 that was missed

MIN_AC_SAS = 1  # Minimum allele count in SAS
temp_sas_filtered = f"{output_dir}/gnomad_chr5_sas_temp.vcf.gz"

# Filter using AC_joint_sas (gnomAD v4 field)
filter_cmd = f"""
bcftools view \
    -i 'INFO/AC_joint_sas >= {MIN_AC_SAS}' \
    -O z \
    -o {temp_sas_filtered} \
    {input_gnomad}
"""

print(f"Filtering variants with AC_joint_sas >= {MIN_AC_SAS}...")
print("This may take 30-60 minutes for a 55GB file...")

result = subprocess.run(filter_cmd, shell=True, capture_output=True, text=True)

if result.returncode != 0:
    print("\n❌ FILTERING FAILED!")
    print(result.stderr)
    raise Exception("SAS filtering failed")


# Count results
count_cmd = f"bcftools view -H {temp_sas_filtered} | wc -l"
result = subprocess.run(count_cmd, shell=True, capture_output=True, text=True)
filtered_count = result.stdout.strip()

print(f"\n✓ Filtering complete!")
print(f"  Filtered variants: {filtered_count}")

filtered_size = os.path.getsize(temp_sas_filtered) / (1024**3)
print(f"  File size: {filtered_size:.2f} GB")


Filtering variants with AC_joint_sas >= 1...
This may take 30-60 minutes for a 55GB file...

✓ Filtering complete!
  Filtered variants: 6412263
  File size: 7.07 GB


In [5]:
temp_sas_filtered = "/content/drive/MyDrive/FYP_DATA/processed_data/gnomad/gnomad_chr5_sas_temp.vcf.gz"

# ============================================================
# PART 3: NORMALIZATION
# ============================================================
print("\n" + "=" * 70)
print("PART 3: NORMALIZING VCF")
print("=" * 70)

output_vcf = f"{output_dir}/gnomad_chr5_sas_normalized.vcf.gz"

# Check if reference needs indexing
if not os.path.exists(f"{reference_fasta}.fai"):
    print("Indexing reference genome...")
    !samtools faidx {reference_fasta}

# Normalize
norm_cmd = f"""
bcftools norm \
    -m- \
    -f {reference_fasta} \
    --check-ref w \
    -O z \
    -o {output_vcf} \
    {temp_sas_filtered}
"""

print("Running normalization...")
print("This may take 20-40 minutes...")

result = subprocess.run(norm_cmd, shell=True, capture_output=True, text=True)

print("\nNormalization statistics:")
print(result.stderr)

if result.returncode != 0:
    print("\n❌ NORMALIZATION FAILED!")
    print(result.stderr)
    raise Exception("Normalization failed")


PART 3: NORMALIZING VCF
Running normalization...
This may take 20-40 minutes...

Normalization statistics:
Lines   total/split/realigned/skipped:	6412263/0/11/0



In [6]:
# Index final file
print("\nIndexing final VCF...")
!bcftools index -t {output_vcf}


Indexing final VCF...


In [ ]:
# !pip install cyvcf2

# !apt-get update
# !apt-get install -y bcftools tabix

# input_gnomad = "/content/drive/MyDrive/FYP_DATA/raw_data/gnomad_data/joint_vcf/chr4.vcf.bgz"

# reference_fasta = "/content/drive/MyDrive/FYP_DATA/raw_data/reference_fasta/hg38_dna.fa"

# output_dir = "/content/drive/MyDrive/FYP_DATA/processed_data/gnomad"


# from cyvcf2 import VCF
# import subprocess
# import os
# import pandas as pd
# import numpy as np
# from cyvcf2 import VCF

# MIN_AC_SAS = 1  # Minimum allele count in SAS
# temp_sas_filtered = f"{output_dir}/gnomad_chr4_sas_temp.vcf.gz"

# # Filter using AC_joint_sas (gnomAD v4 field)
# filter_cmd = f"""
# bcftools view \
#     -i 'INFO/AC_joint_sas >= {MIN_AC_SAS}' \
#     -O z \
#     -o {temp_sas_filtered} \
#     {input_gnomad}
# """

# print(f"Filtering variants with AC_joint_sas >= {MIN_AC_SAS}...")
# print("This may take 30-60 minutes for a 55GB file...")

# result = subprocess.run(filter_cmd, shell=True, capture_output=True, text=True)

# if result.returncode != 0:
#     print("\n❌ FILTERING FAILED!")
#     print(result.stderr)
#     raise Exception("SAS filtering failed")


# # Count results
# count_cmd = f"bcftools view -H {temp_sas_filtered} | wc -l"
# result = subprocess.run(count_cmd, shell=True, capture_output=True, text=True)
# filtered_count = result.stdout.strip()

# print(f"\n✓ Filtering complete!")
# print(f"  Filtered variants: {filtered_count}")

# filtered_size = os.path.getsize(temp_sas_filtered) / (1024**3)
# print(f"  File size: {filtered_size:.2f} GB")


Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [ ]:
!bcftools index /content/drive/MyDrive/FYP_DATA/processed_data/gnomad/gnomad_chr4_sas_temp.vcf.gz
!bcftools view -H /content/drive/MyDrive/FYP_DATA/processed_data/gnomad/gnomad_chr4_sas_temp.vcf.gz | wc -l


^C
^C


In [ ]:
# from cyvcf2 import VCF

# vcf_file = "/content/drive/MyDrive/FYP_DATA/processed_data/gnomad/gnomad_chr4_sas_temp.vcf.gz"
# vcf = VCF(vcf_file)

# # 1️⃣ Count total variants
# total = 0
# for _ in vcf:
#     total += 1
# print(f"Total variants in filtered VCF: {total:,}")


Total variants in filtered VCF: 6,899,540


In [ ]:
# # 2️⃣ Spot-check first 10 variants and AC_joint_sas
# vcf = VCF(vcf_file)  # reset iterator
# for i, variant in enumerate(vcf):
#     if i >= 10:
#         break
#     print(variant.CHROM, variant.POS, variant.REF, variant.ALT, variant.INFO.get('AC_joint_sas', 0))



chr4 10064 CCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACG ['C'] 1
chr4 10147 CCTAACCCTAACCCTAACG ['C'] 1
chr4 10153 CCTAACCCTAACG ['C'] 1
chr4 10156 AACCCTAAC ['A'] 1
chr4 10159 C ['G'] 3
chr4 10159 CCTAACG ['C'] 3
chr4 10163 ACGCT ['A'] 1
chr4 10165 G ['C'] 58
chr4 10165 GCTAACCCTAACC ['G'] 1
chr4 10166 C ['G'] 1


In [ ]:
# # 3️⃣ Ensure all variants have AC_joint_sas >= 1
# vcf = VCF(vcf_file)
# invalid = 0
# for variant in vcf:
#     if variant.INFO.get('AC_joint_sas', 0) < 1:
#         invalid += 1
# print(f"Variants failing AC_joint_sas >= 1: {invalid}")

Variants failing AC_joint_sas >= 1: 0


In [ ]:
# temp_sas_filtered = "/content/drive/MyDrive/FYP_DATA/processed_data/gnomad/gnomad_chr4_sas_temp.vcf.gz"

# # ============================================================
# # PART 3: NORMALIZATION
# # ============================================================
# print("\n" + "=" * 70)
# print("PART 3: NORMALIZING VCF")
# print("=" * 70)

# output_vcf = f"{output_dir}/gnomad_chr4_sas_normalized.vcf.gz"

# # Check if reference needs indexing
# if not os.path.exists(f"{reference_fasta}.fai"):
#     print("Indexing reference genome...")
#     !samtools faidx {reference_fasta}

# # Normalize
# norm_cmd = f"""
# bcftools norm \
#     -m- \
#     -f {reference_fasta} \
#     --check-ref w \
#     -O z \
#     -o {output_vcf} \
#     {temp_sas_filtered}
# """

# print("Running normalization...")
# print("This may take 20-40 minutes...")

# result = subprocess.run(norm_cmd, shell=True, capture_output=True, text=True)

# print("\nNormalization statistics:")
# print(result.stderr)

# if result.returncode != 0:
#     print("\n❌ NORMALIZATION FAILED!")
#     print(result.stderr)
#     raise Exception("Normalization failed")


PART 3: NORMALIZING VCF
Running normalization...
This may take 20-40 minutes...

Normalization statistics:
Lines   total/split/realigned/skipped:	6899540/0/9/0



In [ ]:
# # Index final file
# print("\nIndexing final VCF...")
# !bcftools index -t {output_vcf}


Indexing final VCF...


In [ ]:


# # Final report
# final_size = os.path.getsize(output_vcf) / (1024**3)
# print("\n" + "=" * 70)
# print("🎉 PROCESSING COMPLETE!")
# print("=" * 70)
# print(f"\n📁 Final file: {output_vcf}")
# print(f"📊 Final size: {final_size:.2f} GB")
# print(f"\n✓ Ready for VEP annotation on your WSL desktop!")
# print(f"\n💡 Next: Transfer this file to your desktop and run VEP offline")
# print("=" * 70)


🎉 PROCESSING COMPLETE!

📁 Final file: /content/drive/MyDrive/FYP_DATA/processed_data/gnomad/gnomad_chr4_sas_normalized.vcf.gz
📊 Final size: 7.59 GB

✓ Ready for VEP annotation on your WSL desktop!

💡 Next: Transfer this file to your desktop and run VEP offline


In [ ]:
# # doing for chr3 the previous part was done by bilal only norm by me for chr3

# temp_sas_filtered = "/content/drive/MyDrive/FYP_DATA/processed_data/gnomad/gnomad_chr3_sas_temp.vcf.gz"

# # ============================================================
# # PART 3: NORMALIZATION
# # ============================================================
# print("\n" + "=" * 70)
# print("PART 3: NORMALIZING VCF")
# print("=" * 70)

# output_vcf = f"{output_dir}/gnomad_chr3_sas_normalized.vcf.gz"

# # Check if reference needs indexing
# if not os.path.exists(f"{reference_fasta}.fai"):
#     print("Indexing reference genome...")
#     !samtools faidx {reference_fasta}

# # Normalize
# norm_cmd = f"""
# bcftools norm \
#     -m- \
#     -f {reference_fasta} \
#     --check-ref w \
#     -O z \
#     -o {output_vcf} \
#     {temp_sas_filtered}
# """

# print("Running normalization...")
# print("This may take 20-40 minutes...")

# result = subprocess.run(norm_cmd, shell=True, capture_output=True, text=True)

# print("\nNormalization statistics:")
# print(result.stderr)

# if result.returncode != 0:
#     print("\n❌ NORMALIZATION FAILED!")
#     print(result.stderr)
#     raise Exception("Normalization failed")

# # Index final file
# print("\nIndexing final VCF...")
# !bcftools index -t {output_vcf}


PART 3: NORMALIZING VCF
Running normalization...
This may take 20-40 minutes...

Normalization statistics:
Lines   total/split/realigned/skipped:	7220081/0/12/0


Indexing final VCF...
